# 🔨 Data Generation: Источники данных для ТК РФ

Этот notebook для исследования различных источников данных и разработки парсеров для загрузки статей Трудового кодекса РФ.

## Цели:
1. 🔍 Исследовать доступные источники данных
2. 🧪 Протестировать доступность и качество данных
3. ⚙️ Создать парсеры для разных форматов
4. 📊 Сравнить источники и выбрать оптимальный
5. 💾 Сгенерировать JSON файлы для загрузки в БД

## 1. Setup: Импорт библиотек

In [1]:
import os
import sys
import json
import re
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, asdict
from urllib.parse import urljoin, urlparse

# HTTP клиенты
import httpx
from bs4 import BeautifulSoup

# Для визуализации
import pandas as pd
from IPython.display import display, HTML, Markdown

# Добавляем корневую директорию проекта в PYTHONPATH
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("✅ Библиотеки импортированы")
print(f"📂 Корень проекта: {project_root}")

✅ Библиотеки импортированы
📂 Корень проекта: /home/user/Nextcloud/Projects/home/legal_agent


## 2. Структура данных для статей

Определяем единую структуру данных для статей из любого источника.

In [2]:
@dataclass
class ArticleSource:
    """Метаданные о источнике статьи"""
    source_type: str  # 'consultant', 'pravo_gov', 'garant', etc.
    source_url: str
    fetch_date: str  # ISO format
    version: Optional[str] = None
    
@dataclass 
class ArticleData:
    """Структура статьи ТК РФ"""
    number: str  # "80", "81", "84.1", etc.
    title: str
    content: str
    chapter: Optional[str] = None  # "Глава 13. Прекращение трудового договора"
    section: Optional[str] = None  # "Раздел III. Трудовой договор"
    source: Optional[ArticleSource] = None
    
    def to_dict(self) -> dict:
        """Конвертация в словарь для JSON"""
        data = {
            "number": self.number,
            "title": self.title,
            "content": self.content,
            "chapter": self.chapter,
            "section": self.section,
        }
        if self.source:
            data["source"] = asdict(self.source)
        return data

# Пример
example = ArticleData(
    number="80",
    title="Расторжение трудового договора по инициативе работника",
    content="Работник имеет право расторгнуть трудовой договор...",
    chapter="Глава 13. Прекращение трудового договора",
    source=ArticleSource(
        source_type="test",
        source_url="http://example.com",
        fetch_date=datetime.now().isoformat()
    )
)

print("📋 Пример структуры данных:")
print(json.dumps(example.to_dict(), ensure_ascii=False, indent=2))

📋 Пример структуры данных:
{
  "number": "80",
  "title": "Расторжение трудового договора по инициативе работника",
  "content": "Работник имеет право расторгнуть трудовой договор...",
  "chapter": "Глава 13. Прекращение трудового договора",
  "section": null,
  "source": {
    "source_type": "test",
    "source_url": "http://example.com",
    "fetch_date": "2025-11-23T21:19:42.398477",
    "version": null
  }
}


## 3. Источник #1: КонсультантПлюс (consultant.ru)

Самый популярный источник юридической информации в России.

In [5]:
# URL Трудового кодекса на КонсультантПлюс
CONSULTANT_TK_URL = "https://www.consultant.ru/document/cons_doc_LAW_34683/"

async def fetch_consultant_tk():
    """Загружаем страницу ТК РФ с КонсультантПлюс"""
    async with httpx.AsyncClient(timeout=30.0, trust_env=False, follow_redirects=True) as client:
        try:
            response = await client.get(CONSULTANT_TK_URL)
            response.raise_for_status()
            
            print(f"✅ Успешно загружено с {CONSULTANT_TK_URL}")
            print(f"📊 Размер: {len(response.text)} символов")
            print(f"📄 Encoding: {response.encoding}")
            
            return response.text
        except Exception as e:
            print(f"❌ Ошибка загрузки: {e}")
            return None

# Загружаем
consultant_html = await fetch_consultant_tk()

✅ Успешно загружено с https://www.consultant.ru/document/cons_doc_LAW_34683/
📊 Размер: 164960 символов
📄 Encoding: utf-8


In [6]:
def parse_consultant_structure(html: str) -> Dict:
    """Анализируем структуру HTML КонсультантПлюс"""
    soup = BeautifulSoup(html, 'html.parser')
    
    info = {
        "title": None,
        "main_content_tags": [],
        "article_patterns": [],
        "structure_elements": []
    }
    
    # Ищем заголовок
    title = soup.find('title')
    if title:
        info["title"] = title.get_text(strip=True)
    
    # Ищем основной контент
    for tag in ['article', 'div', 'section']:
        elements = soup.find_all(tag, class_=re.compile(r'(content|article|document)'))
        if elements:
            info["main_content_tags"].append(f"{tag}: {len(elements)} elements")
    
    # Ищем паттерны статей ("Статья 80", "Статья 81" и т.д.)
    text = soup.get_text()
    article_matches = re.findall(r'Статья\s+(\d+(?:\.\d+)?)', text)
    info["article_patterns"] = list(set(article_matches))[:10]  # Первые 10 уникальных
    
    # Ищем структурные элементы (главы, разделы)
    for pattern in [r'Глава\s+\d+', r'Раздел\s+[IVX]+']:
        matches = re.findall(pattern, text)
        if matches:
            info["structure_elements"].extend(list(set(matches))[:5])
    
    return info

if consultant_html:
    structure = parse_consultant_structure(consultant_html)
    print("📊 Структура страницы КонсультантПлюс:")
    print(json.dumps(structure, ensure_ascii=False, indent=2))

📊 Структура страницы КонсультантПлюс:
{
  "title": "\"Трудовой кодекс Российской Федерации\" (ТК РФ) от 30.12.2001 N 197-ФЗ (последняя редакция) \\ КонсультантПлюс",
  "main_content_tags": [
    "div: 15 elements",
    "section: 1 elements"
  ],
  "article_patterns": [
    "273",
    "214",
    "294",
    "42",
    "41",
    "383",
    "119",
    "292",
    "378",
    "374"
  ],
  "structure_elements": [
    "Глава 38",
    "Глава 47",
    "Глава 39",
    "Глава 34",
    "Глава 54",
    "Раздел XIV",
    "Раздел VII",
    "Раздел XI",
    "Раздел X",
    "Раздел I"
  ]
}


### Парсер для КонсультантПлюс

**⚠️ Примечание:** КонсультантПлюс показывает оглавление вместо полного текста статей на главной странице. Для получения полного текста статей потребуется:
1. Парсить каждую статью отдельно (переходить по ссылкам)
2. Или искать другой API/источник с полным текстом
3. Или использовать готовые датасеты

Для production рекомендуется разработать более сложный парсер или найти альтернативный источник данных.

In [14]:
def parse_consultant_articles(html: str, limit: int = 5) -> List[ArticleData]:
    """
    Парсим статьи с КонсультантПлюс (прототип)
    
    ОГРАНИЧЕНИЕ: Эта функция - базовый прототип. КонсультантПлюс показывает
    оглавление на главной странице, а не полный текст статей. Для production
    требуется более сложный подход:
    - Парсинг отдельных страниц каждой статьи
    - Использование альтернативных источников данных
    """
    soup = BeautifulSoup(html, 'html.parser')
    articles = []
    
    # Пытаемся извлечь хотя бы базовую информацию из оглавления
    text = soup.get_text()
    
    # Паттерн: "Статья 80. Название"
    pattern = r'Статья\s+(\d+(?:\.\d+)?)\.\s+([^\n]+?)(?=\n|$)'
    matches = list(re.finditer(pattern, text))
    
    print(f"🔍 Найдено упоминаний статей в HTML: {len(matches)}")
    
    for i, match in enumerate(matches[:limit]):
        number = match.group(1)
        title = match.group(2).strip()
        
        # Пытаемся найти контент между текущей статьей и следующей
        start_pos = match.end()
        if i + 1 < len(matches):
            end_pos = matches[i + 1].start()
        else:
            end_pos = start_pos + 1000
        
        content = text[start_pos:end_pos].strip()
        
        # Очищаем контент
        content = re.sub(r'\n\s*\n', '\n\n', content)
        content = content[:500]
        
        # Проверяем, что это не просто список других статей
        if content and not content.startswith("Статья"):
            article = ArticleData(
                number=number,
                title=title,
                content=content,
                source=ArticleSource(
                    source_type="consultant",
                    source_url=CONSULTANT_TK_URL,
                    fetch_date=datetime.now().isoformat()
                )
            )
            articles.append(article)
    
    return articles

if consultant_html:
    print("⚠️  ВНИМАНИЕ: КонсультантПлюс показывает оглавление, а не полный текст")
    print("   Для реальных данных используйте синтетический датасет (секция 6) или")
    print("   разработайте более сложный парсер с переходом по ссылкам статей.\n")
    
    consultant_articles = parse_consultant_articles(consultant_html, limit=5)
    print(f"✅ Извлечено записей из оглавления: {len(consultant_articles)}\n")
    
    if consultant_articles:
        for i, article in enumerate(consultant_articles, 1):
            print(f"📄 {i}. Статья {article.number}: {article.title}")
            print(f"   Контент ({len(article.content)} символов): {article.content[:150]}...")
            print()
    else:
        print("⚠️  Не удалось извлечь полноценные статьи (только оглавление)")
        print("   Используйте синтетические данные для разработки (см. секцию 6)")

⚠️  ВНИМАНИЕ: КонсультантПлюс показывает оглавление, а не полный текст
   Для реальных данных используйте синтетический датасет (секция 6) или
   разработайте более сложный парсер с переходом по ссылкам статей.

🔍 Найдено упоминаний статей в HTML: 1
✅ Извлечено записей из оглавления: 1

📄 1. Статья 1: Цели и задачи трудового законодательстваСтатья 2. Основные принципы правового регулирования трудовых отношений и иных непосредственно связанных с ними отношенийСтатья 3. Запрещение дискриминации в сфере трудаСтатья 4. Запрещение принудительного трудаСтатья 5. Трудовое законодательство и иные акты, содержащие нормы трудового праваСтатья 6. Разграничение полномочий между федеральными органами государственной власти и органами государственной власти субъектов Российской Федерации в сфере трудовых отношений и иных непосредственно связанных с ними отношенийСтатья 7. Утратила силуСтатья 8. Локальные нормативные акты, содержащие нормы трудового праваСтатья 9. Регулирование трудовых отношений и и

## 4. Источник #2: Официальный портал pravo.gov.ru

Официальный интернет-портал правовой информации.

In [13]:
# Нужно найти прямую ссылку на ТК РФ
PRAVO_GOV_SEARCH = "http://publication.pravo.gov.ru/"

async def explore_pravo_gov():
    """Исследуем структуру pravo.gov.ru"""
    async with httpx.AsyncClient(timeout=30.0, trust_env=False, follow_redirects=True) as client:
        try:
            response = await client.get(PRAVO_GOV_SEARCH)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            print(f"✅ Успешно загружено с {PRAVO_GOV_SEARCH}")
            print(f"📊 Title: {soup.title.get_text() if soup.title else 'N/A'}")
            
            # Ищем формы поиска
            forms = soup.find_all('form')
            print(f"\n🔍 Найдено форм: {len(forms)}")
            
            # Ищем ссылки на документы
            links = soup.find_all('a', href=re.compile(r'(document|doc)'))
            print(f"🔗 Ссылок на документы: {len(links)}")
            
            if links:
                print("\nПримеры ссылок:")
                for link in links[:5]:
                    print(f"  - {link.get_text(strip=True)}: {link['href']}")
            
            return response.text
        except Exception as e:
            print(f"❌ Ошибка: {e}")
            return None

# Исследуем
pravo_html = await explore_pravo_gov()

✅ Успешно загружено с http://publication.pravo.gov.ru/
📊 Title: Официальное опубликование правовых актов

🔍 Найдено форм: 0
🔗 Ссылок на документы: 13

Примеры ссылок:
  - Сегодня: /documents/daily
  - Неделя: /documents/weekly
  - Месяц: /documents/monthly
  - Сегодня: /documents/daily
  - Неделя: /documents/weekly
